# AF2 parent-residual seed-42 decisions
Membandingkan SAF1 dengan SAF0 dan IGEM1 dengan IGEM0. Tidak ada training dan tidak mengakses test.

In [ ]:
BRANCH='codex/af2-parent-residual'
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys, time, json
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    time.sleep(2)
else: raise RuntimeError('Git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root
from coffee_detector.experiments.run_faruq_v3_af2_parent_residual_decision import run_faruq_v3_af2_parent_residual_decision
PROJECT=resolve_drive_project_root(required_relative_paths=('experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt',))
OUTPUT=PROJECT/'experiments/faruq-v3-af2-parent-residual-v1'; REPORTS=OUTPUT/'val_reports'
print('PROJECT:',PROJECT)

In [ ]:
decisions={}
for family,control,candidate in (('saf','AF2SAF0','AF2SAF1'),('igem','AF2IGEM0','AF2IGEM1')):
    control_path=REPORTS/f'{control}_seed42_result.json'; candidate_path=REPORTS/f'{candidate}_seed42_result.json'
    if not control_path.is_file() or not candidate_path.is_file():
        print(f'{family.upper()} BELUM LENGKAP:',control_path.is_file(),candidate_path.is_file()); continue
    decision=run_faruq_v3_af2_parent_residual_decision(family,control_path,candidate_path,REPORTS/f'{family}_parent_residual_seed42_decision.json')
    decisions[family]=decision
    print('\n',family.upper(),decision['decision'])
    print(json.dumps(decision,indent=2,ensure_ascii=False))
assert decisions,'Belum ada satu family lengkap untuk dinilai.'
print('\nTRAINING: False | TEST: False')